In [1]:
import numpy as np
import time
import os
import ase
from pyscf import gto, dft, df, lib
from pyscf.scf import hf
import scipy
from equiv_dens.utils import base as utils
%cd ..
hf.MUTE_CHKFILE = True
%load_ext autoreload
%autoreload 2

loading config file None
/home/mihail/Documents/workspace/equiv_dens


In [ ]:
# mol = gto.M(atom='O  0  0  0.1184; H  0,  0.7532, -0.4735; H 0,  -0.7532, -0.4735 ', basis='def2svp')
# mf = dft.RKS(mol)
# mf.chkfile=False
# mf.xc = 'pbe'
# mf.kernel()
# g = mf.nuc_grad_method()
# g.kernel()
data = np.load('datasets/h2o_dynamic_centered.npy', allow_pickle=True).item()
basis = 'augccpvdz'
auxbasis = 'augccpvqzjkfit'

atom_types = data['atom_types']

print(len(data['positions']))
save_path = 'datasets/h2o_dynamic_augccpvdz_df_augccpvqzjkfit.npy'
npy_path = 'datasets/h2o_dynamic_augccpvdz.npy'
if os.path.exists(save_path):
    results = list(np.load(save_path, allow_pickle=True))
else:
    results = []
print('results len', len(results))
for i in range(len(results), len(data['positions'])):
    print('calc', i)
    start = time.time()
    pos = data['positions'][i]
    atom = []
    for j in range(len(atom_types)):
        atom.append((atom_types[j], pos[j, :])) 
    mol = gto.M(atom=atom, basis=basis)
    #print(mol.pack())
    mf = dft.RKS(mol)
    mf.chkfile=False
    mf.xc = 'pbe'
    mf.kernel()
    g = mf.nuc_grad_method()
    gradients = g.grad()
    print('elapsed', time.time() - start)
    #print(mfs[i].mo_coeff)
    res = []
    res.append(mol.pack())
    calc_dict = {}
    calc_dict['mo_coeff'] = mf.mo_coeff
    calc_dict['mo_occ'] = mf.mo_occ
    calc_dict['energy'] = mf.e_tot
    calc_dict['forces'] = -gradients/ase.units.Bohr

    dm1 = mf.make_rdm1(mf.mo_coeff, mf.mo_occ)
    auxmol = df.addons.make_auxmol(mol, auxbasis)

    ints_3c2e = df.incore.aux_e2(mol, auxmol, intor='int3c2e')
    ints_2c2e = auxmol.intor('int2c2e')
    print('ints3c2e shape', ints_3c2e.shape)
    print('ints2c2e shape', ints_2c2e.shape)

    nao = mol.nao
    naux = auxmol.nao
    df_coef = scipy.linalg.solve(ints_2c2e, ints_3c2e.reshape(nao*nao, naux).T)
    df_coef = df_coef.reshape(naux, nao, nao)
    if dm1.ndim > 2:
        df_basis = []
        for j in range(dm1.shape[0]):
            df_basis.append(lib.einsum('Pij,ij->P', df_coef, dm1[j]))
        df_basis = np.stack(df_basis, axis=0)
        print(df_basis.shape)

    else:
        df_basis = lib.einsum('Pij,ij->P', df_coef, dm1)

    calc_dict['df_coeff'] = df_basis
    calc_dict['auxbasis'] = auxbasis
    res.append(calc_dict)
    results.append(res)

    if i%100 == 0:
        np.save(save_path, results, allow_pickle=True)
np.save(save_path, results, allow_pickle=True)
npy_data = utils.calc_dict_to_npy(results, convert_forces=False, compress_atoms=False)
np.save(npy_path, npy_data, allow_pickle=True)

In [ ]:
# calculating for a single molecule, and getting different energy components
data = np.load('datasets/h2o_dynamic_centered.npy', allow_pickle=True).item()
basis = 'augccpvdz'

atom_types = data['atom_types']

i = 0
print('calc', i)
start = time.time()
pos = data['positions'][i]
atom = []
for j in range(len(atom_types)):
    atom.append((atom_types[j], pos[j, :]))
mol = gto.M(atom=atom, basis=basis)
#print(mol.pack())
mf = hf.RHF(mol)
mf.max_cycle = 0
mf.init_guess = 'atom'
mf.chkfile=False
mf.kernel()
print(mf.energy_tot())
print(mf.e_tot)

In [ ]:
dm = mf.make_rdm1()
m_kin = mol.intor('int1e_kin')
m_nuc = mol.intor('int1e_nuc')

e_kin = np.einsum('ij,ji', dm, m_kin)
e_nuc = np.einsum('ij,ji', dm, m_nuc)

veff = mf.get_veff()
ecoul = veff.ecoul
exc = veff.exc

print('total energy', e_kin + e_nuc + ecoul + exc + mf.energy_nuc())
print(mf.__dir__())
print('total_energy etot', mf.e_tot)
print('total_energy', mf.energy_tot())
print('nuclear energy', mf.energy_nuc())
print('eletronic energy, coulomb energy', mf.energy_elec())
print(mf.get_veff().shape)
print(mf.mo_coeff.shape)

In [ ]:
def get_energy_components(mol, mf):
    """
    Get energy components for a single molecule.

    Args:
        mol: pyscf molecule
        mf: pyscf scf object
    Returns:
        energies: dictionary of energy components
    """
    dm = mf.make_rdm1()
    m_kin = mol.intor('int1e_kin')
    m_nuc = mol.intor('int1e_nuc')
    h1e = mf.get_hcore()
    veff = mf.get_veff()

    energies = {}
    energies['energy'] = mf.energy_tot()
    energies['energy_e_kin'] = np.einsum('ij,ji', dm, m_kin)
    energies['energy_e_nuc'] = np.einsum('ij,ji', dm, m_nuc)
    energies['energy_coul'] = veff.ecoul
    energies['energy_exc'] = veff.exc
    energies['energy_nuc'] = mf.energy_nuc()
    # print('energies', energies)
    # print('total energy', energies['energy'])
    # print('mf energy elec', mf.energy_elec())
    # print('mf energy nuc', mf.energy_nuc())
    # print('mf energy elec + nuc', mf.energy_elec() + mf.energy_nuc())
    # print('mf ecoul', energies['energy_coul'] + energies['energy_exc'])
    # print('energy h1e', energies['energy_e_kin'] + energies['energy_e_nuc'])
    # print('mf h1e', np.einsum('ij,ji', dm, h1e))
    #
    # print('total elec', energies['energy_e_kin'] + energies['energy_e_nuc'] +
    #       energies['energy_coul'] + energies['energy_exc'])
    # print('mf elec', np.einsum('ij,ji', dm, h1e) + energies['energy_coul'] + energies['energy_exc'])
    # print('summed components', energies['energy_e_kin'] + energies['energy_e_nuc'] +
    #       energies['energy_coul'] + energies['energy_exc'] + energies['energy_nuc'])

    assert np.isclose(energies['energy'], energies['energy_e_kin'] + energies['energy_e_nuc'] +
                      energies['energy_coul'] + energies['energy_exc'] + energies['energy_nuc'])
    return energies

In [8]:
set_types = ['train', 'valid', 'test']
for set_type in set_types:
    data = np.load('datasets/h2o_small_' + set_type + '_augccpvdz.npy', allow_pickle=True).item()
    basis = 'augccpvdz'
    auxbasis = 'augccpvqzjkfit'

    print(len(data['positions']))
    save_path = 'datasets/h2o_small_' + set_type + '_dft_augccpvdz_energy_comps_calc.npy'
    npy_path = 'datasets/h2o_small_' + set_type + '_dft_augccpvdz_energy_comps.npy'
    if os.path.exists(save_path):
        results = list(np.load(save_path, allow_pickle=True))
    else:
        results = []
    print('results len', len(results))
    for i in range(len(results), len(data['positions'])):
        print('calc', i)
        start = time.time()
        print('data positions shape', data['positions'].shape)
        pos = data['positions'][i]
        anums = data['atom_numbers'][i]
        print('pos shape', pos.shape)
        atom = []
        for j in range(len(anums)):
            atom.append((anums[j], pos[j, :])) 
        mol = gto.M(atom=atom, basis=basis)
        res = []
        res.append(mol.pack())
        #print(mol.pack())
        mf = dft.RKS(mol)
        mf.init_guess = 'atom'
        mf.max_cycle = 0
        mf.chkfile=False
        mf.xc = 'pbe'
        mf.kernel()
        g = mf.nuc_grad_method()
        gradients = g.grad()
        energies_SAD = get_energy_components(mol, mf)
        energies_SAD = {k + '_SAD': v for k, v in energies_SAD.items()}
        calc_dict = {}
        calc_dict.update(energies_SAD)
        calc_dict['forces_SAD'] = -gradients/ase.units.Bohr
        mol = gto.M(atom=atom, basis=basis)
        #print(mol.pack())
        mf = dft.RKS(mol)
        mf.chkfile=False
        mf.xc = 'pbe'
        mf.kernel()
        g = mf.nuc_grad_method()
        gradients = g.grad()
        energies = get_energy_components(mol, mf)

        calc_dict['forces'] = -gradients/ase.units.Bohr
        calc_dict.update(energies)

        print('calc_dict', calc_dict)
        res.append(calc_dict)
        results.append(res)

        if i%10 == 0:
            np.save(save_path, results, allow_pickle=True)
    np.save(save_path, results, allow_pickle=True)
    npy_data = utils.calc_dict_to_npy(results, convert_forces=False, compress_atoms=False)
    npy_data_compressed = utils.calc_dict_to_npy(results, convert_forces=False, compress_atoms=True)
    print('atom_number nc', npy_data['atom_numbers'][:3])
    print('atom_number c', npy_data_compressed['atom_numbers'][:3])
    print('pos nc', npy_data['positions'][:3])
    print('pos c', npy_data_compressed['positions'][:3])
    print('forces nc', npy_data['forces'][:3])
    print('forces c', npy_data_compressed['forces'][:3])
    print('forces sad nc', npy_data['forces_SAD'][:3])
    print('forces sad c', npy_data_compressed['forces_SAD'][:3])
    np.save(npy_path, npy_data, allow_pickle=True)

100
results len 0
calc 0
converged SCF energy = -76.3400734696848
calc 1
converged SCF energy = -76.357571836433
calc 2
converged SCF energy = -76.3216300513999
calc 3
converged SCF energy = -76.3301241150047
calc 4
converged SCF energy = -76.3575156401553
calc 5
converged SCF energy = -76.3489200440544
calc 6
converged SCF energy = -76.354418770028
calc 7
converged SCF energy = -76.3382051551203
calc 8
converged SCF energy = -76.3306062649903
calc 9
converged SCF energy = -76.3116804906144
calc 10
converged SCF energy = -76.3540169794951
calc 11
converged SCF energy = -76.321613673482
calc 12
converged SCF energy = -76.3132378254684
calc 13
converged SCF energy = -76.3268942587044
calc 14
converged SCF energy = -76.3591715427401
calc 15
converged SCF energy = -76.3401323466418
calc 16
converged SCF energy = -76.3364041493276
calc 17
converged SCF energy = -76.3512802050817
calc 18
converged SCF energy = -76.3366110706739
calc 19
converged SCF energy = -76.3168892440311
calc 20
converg

calc 62
converged SCF energy = -76.3421387008564
calc 63
converged SCF energy = -76.3444510939799
calc 64
converged SCF energy = -76.344240150403
calc 65
converged SCF energy = -76.3407385731545
calc 66
converged SCF energy = -76.3455792142957
calc 67
converged SCF energy = -76.3432698697042
calc 68
converged SCF energy = -76.3444751329801
calc 69
converged SCF energy = -76.3263131275186
calc 70
converged SCF energy = -76.3451856179845
calc 71
converged SCF energy = -76.3538705499209
calc 72
converged SCF energy = -76.3340275073657
calc 73
converged SCF energy = -76.3401195988275
calc 74
converged SCF energy = -76.3508744498065
calc 75
converged SCF energy = -76.3197035445648
calc 76
converged SCF energy = -76.3539526243354
calc 77
converged SCF energy = -76.3092209274349
calc 78
converged SCF energy = -76.3451200214197
calc 79
converged SCF energy = -76.3376486915115
calc 80
converged SCF energy = -76.3143214091739
calc 81
converged SCF energy = -76.3149005704183
calc 82
converged SCF

In [9]:
np.save(save_path, results, allow_pickle=True)

In [25]:
set_types = ['train', 'valid', 'test']
for set_type in set_types:
    data = np.load('datasets/h2o_small_' + set_type + '_augccpvdz_df_augccpvqzjkfit.npy', allow_pickle=True)
    basis = 'augccpvdz'
    auxbasis = 'augccpvqzjkfit'

    print(len(data))
    save_path = 'datasets/h2o_small_' + set_type + '_dft_augccpvdz_df_hm_dm_oe_calc.npy'
    npy_path = 'datasets/h2o_small_' + set_type + '_dft_augccpvdz_df_hm_dm_oe.npy'
    if os.path.exists(save_path):
        results = list(np.load(save_path, allow_pickle=True))
    else:
        results = []
    print('results len', len(results))
    for i in range(len(results), len(data)):
        print('calc', i)
        start = time.time()
        atom = data[i][0]["atom"] 
        mol = gto.M(atom=atom, basis=basis)
        res = []
        res.append(mol.pack())
        mf = dft.RKS(mol)
        mf.chkfile=False
        mf.xc = 'pbe'
        mf.kernel()
        g = mf.nuc_grad_method()
        gradients = g.grad()
        print('elapsed', time.time() - start)
        #print(mfs[i].mo_coeff)
        res = []
        res.append(mol.pack())
        calc_dict = {}
        calc_dict['mo_coeff'] = mf.mo_coeff
        print('mo_coeff shape', calc_dict['mo_coeff'].shape)
        calc_dict['mo_occ'] = mf.mo_occ
        calc_dict['energy'] = mf.e_tot
        calc_dict['forces'] = -gradients/ase.units.Bohr

        dm1 = mf.make_rdm1(mf.mo_coeff, mf.mo_occ)
        auxmol = df.addons.make_auxmol(mol, auxbasis)

        ints_3c2e = df.incore.aux_e2(mol, auxmol, intor='int3c2e')
        ints_2c2e = auxmol.intor('int2c2e')
        print('ints3c2e shape', ints_3c2e.shape)
        print('ints2c2e shape', ints_2c2e.shape)

        nao = mol.nao
        naux = auxmol.nao
        df_coef = scipy.linalg.solve(ints_2c2e, ints_3c2e.reshape(nao*nao, naux).T)
        df_coef = df_coef.reshape(naux, nao, nao)
        if dm1.ndim > 2:
            df_basis = []
            for j in range(dm1.shape[0]):
                df_basis.append(lib.einsum('Pij,ij->P', df_coef, dm1[j]))
            df_basis = np.stack(df_basis, axis=0)
            print(df_basis.shape)

        else:
            df_basis = lib.einsum('Pij,ij->P', df_coef, dm1)

        calc_dict['df_coeff'] = df_basis
        calc_dict['auxbasis'] = auxbasis
        # print('calc_dict', calc_dict)
        oe = mf.mo_energy
        hm = hf.get_fock(mf)
        calc_dict.update({"mo_energies":oe, "density_matrix": dm1,
                          "hamiltonian_matrix": hm})
        res.append(calc_dict)
        results.append(res)
        if i%10 == 0:
            np.save(save_path, results, allow_pickle=True)
    np.save(save_path, results, allow_pickle=True)
    npy_data = utils.calc_dict_to_npy(results, convert_forces=False, compress_atoms=False)
    npy_data_compressed = utils.calc_dict_to_npy(results, convert_forces=False, compress_atoms=True)
    # print('atom_number nc', npy_data['atom_numbers'][:3])
    # print('atom_number c', npy_data_compressed['atom_numbers'][:3])
    # print('pos nc', npy_data['positions'][:3])
    # print('pos c', npy_data_compressed['positions'][:3])
    # print('forces nc', npy_data['forces'][:3])
    # print('forces c', npy_data_compressed['forces'][:3])
    # print('forces sad nc', npy_data['forces_SAD'][:3])
    # print('forces sad c', npy_data_compressed['forces_SAD'][:3])
    np.save(npy_path, npy_data, allow_pickle=True)

100
results len 100
mo_coeff
mo_occ
energy
key has energy or forces
forces
key has energy or forces
df_coeff
auxbasis
mo_energies
density_matrix
hamiltonian_matrix
mo_coeff
mo_occ
energy
key has energy or forces
forces
key has energy or forces
df_coeff
auxbasis
mo_energies
density_matrix
hamiltonian_matrix
100
results len 100
mo_coeff
mo_occ
energy
key has energy or forces
forces
key has energy or forces
df_coeff
auxbasis
mo_energies
density_matrix
hamiltonian_matrix
mo_coeff
mo_occ
energy
key has energy or forces
forces
key has energy or forces
df_coeff
auxbasis
mo_energies
density_matrix
hamiltonian_matrix
100
results len 0
calc 0
converged SCF energy = -76.3521007799265
--------------- RKS gradients ---------------
         x                y                z
0 O     0.0103023008    -0.0826316891     0.0513632336
1 H     0.0255276317     0.0170360189     0.0018363988
2 H    -0.0358299430     0.0655957181    -0.0532026488
----------------------------------------------
elapsed 0.91919

elapsed 0.9326901435852051
mo_coeff shape (41, 41)
ints3c2e shape (41, 41, 294)
ints2c2e shape (294, 294)
calc 17
converged SCF energy = -76.3201575933143
--------------- RKS gradients ---------------
         x                y                z
0 O     0.0208838482    -0.0993321776    -0.1055884280
1 H    -0.0790123231    -0.0026076263     0.0301995779
2 H     0.0581320867     0.1019447342     0.0753888577
----------------------------------------------
elapsed 0.9362006187438965
mo_coeff shape (41, 41)
ints3c2e shape (41, 41, 294)
ints2c2e shape (294, 294)
calc 18
converged SCF energy = -76.3377686138713
--------------- RKS gradients ---------------
         x                y                z
0 O     0.0208442895     0.0494037995     0.0643227973
1 H     0.0078991324    -0.0399741471    -0.0161582776
2 H    -0.0287452447    -0.0094280672    -0.0481664267
----------------------------------------------
elapsed 0.8907153606414795
mo_coeff shape (41, 41)
ints3c2e shape (41, 41, 294)
ints

elapsed 0.8537144660949707
mo_coeff shape (41, 41)
ints3c2e shape (41, 41, 294)
ints2c2e shape (294, 294)
calc 35
converged SCF energy = -76.3030883166089
--------------- RKS gradients ---------------
         x                y                z
0 O    -0.0208234411     0.1401497605    -0.0014373239
1 H    -0.0183352448    -0.1134779835    -0.0051357102
2 H     0.0391604417    -0.0266698766     0.0065704300
----------------------------------------------
elapsed 0.9290564060211182
mo_coeff shape (41, 41)
ints3c2e shape (41, 41, 294)
ints2c2e shape (294, 294)
calc 36
converged SCF energy = -76.3541184637851
--------------- RKS gradients ---------------
         x                y                z
0 O    -0.0016271327    -0.0717165561     0.0166446717
1 H     0.0210488934     0.0402582359    -0.0294896864
2 H    -0.0194176868     0.0314592159     0.0128453393
----------------------------------------------
elapsed 0.8739621639251709
mo_coeff shape (41, 41)
ints3c2e shape (41, 41, 294)
ints

elapsed 0.8936901092529297
mo_coeff shape (41, 41)
ints3c2e shape (41, 41, 294)
ints2c2e shape (294, 294)
calc 53
converged SCF energy = -76.3314019260291
--------------- RKS gradients ---------------
         x                y                z
0 O    -0.0194535822    -0.0807224877     0.0036294520
1 H    -0.0127623103     0.0872988277    -0.0029308530
2 H     0.0322142544    -0.0065767886    -0.0007000920
----------------------------------------------
elapsed 0.9313333034515381
mo_coeff shape (41, 41)
ints3c2e shape (41, 41, 294)
ints2c2e shape (294, 294)
calc 54
converged SCF energy = -76.3424561260145
--------------- RKS gradients ---------------
         x                y                z
0 O    -0.0294373061     0.0468096138    -0.0754197369
1 H     0.0591683411    -0.0646189294     0.0939012151
2 H    -0.0297303146     0.0178099597    -0.0184852604
----------------------------------------------
elapsed 0.8643434047698975
mo_coeff shape (41, 41)
ints3c2e shape (41, 41, 294)
ints

elapsed 0.8881828784942627
mo_coeff shape (41, 41)
ints3c2e shape (41, 41, 294)
ints2c2e shape (294, 294)
calc 71
converged SCF energy = -76.3545123437664
--------------- RKS gradients ---------------
         x                y                z
0 O     0.0355272489     0.0043862399     0.0288452901
1 H    -0.0221387823    -0.0324824997     0.0194498514
2 H    -0.0133909498     0.0280967125    -0.0482898471
----------------------------------------------
elapsed 0.8765614032745361
mo_coeff shape (41, 41)
ints3c2e shape (41, 41, 294)
ints2c2e shape (294, 294)
calc 72
converged SCF energy = -76.3247212362162
--------------- RKS gradients ---------------
         x                y                z
0 O    -0.0577473622    -0.0345339842     0.0299632478
1 H     0.0235427740     0.0145101163    -0.0139720358
2 H     0.0342012147     0.0200259542    -0.0159956353
----------------------------------------------
elapsed 0.9270384311676025
mo_coeff shape (41, 41)
ints3c2e shape (41, 41, 294)
ints

elapsed 0.9050853252410889
mo_coeff shape (41, 41)
ints3c2e shape (41, 41, 294)
ints2c2e shape (294, 294)
calc 89
converged SCF energy = -76.3021302326116
--------------- RKS gradients ---------------
         x                y                z
0 O    -0.0811100357     0.1142719099    -0.1421100617
1 H     0.0361145044    -0.0970917893     0.0547551603
2 H     0.0449934380    -0.0171789232     0.0873508530
----------------------------------------------
elapsed 0.9787125587463379
mo_coeff shape (41, 41)
ints3c2e shape (41, 41, 294)
ints2c2e shape (294, 294)
calc 90
converged SCF energy = -76.3520341895233
--------------- RKS gradients ---------------
         x                y                z
0 O    -0.0246553351     0.0811573540    -0.0391175368
1 H     0.0098084291    -0.0874894902     0.0177073782
2 H     0.0148509519     0.0063356025     0.0214123751
----------------------------------------------
elapsed 0.8752377033233643
mo_coeff shape (41, 41)
ints3c2e shape (41, 41, 294)
ints

In [26]:
data = np.load('datasets/h2o_static_pyscf_dft.npy', allow_pickle=True)
basis = 'augccpvdz'
auxbasis = 'augccpvqzjkfit'

In [27]:
print(len(data))
save_path = 'datasets/h2o_static_pyscf_dft_augccpvdz_df_hm_dm_oe_calc.npy'
npy_path = 'datasets/h2o_static_pyscf_dft_augccpvdz_df_hm_dm_oe.npy'
if os.path.exists(save_path):
    results = list(np.load(save_path, allow_pickle=True))
else:
    results = []
print('results len', len(results))
for i in range(len(results), len(data)):
    print('calc', i)
    start = time.time()
    atom = data[i][0]["atom"] 
    mol = gto.M(atom=atom, basis=basis)
    res = []
    res.append(mol.pack())
    mf = dft.RKS(mol)
    mf.chkfile=False
    mf.xc = 'pbe'
    mf.kernel()
    g = mf.nuc_grad_method()
    gradients = g.grad()
    print('elapsed', time.time() - start)
    #print(mfs[i].mo_coeff)
    res = []
    res.append(mol.pack())
    calc_dict = {}
    calc_dict['mo_coeff'] = mf.mo_coeff
    print('mo_coeff shape', calc_dict['mo_coeff'].shape)
    calc_dict['mo_occ'] = mf.mo_occ
    calc_dict['energy'] = mf.e_tot
    calc_dict['forces'] = -gradients/ase.units.Bohr

    dm1 = mf.make_rdm1(mf.mo_coeff, mf.mo_occ)
    auxmol = df.addons.make_auxmol(mol, auxbasis)

    ints_3c2e = df.incore.aux_e2(mol, auxmol, intor='int3c2e')
    ints_2c2e = auxmol.intor('int2c2e')
    print('ints3c2e shape', ints_3c2e.shape)
    print('ints2c2e shape', ints_2c2e.shape)

    nao = mol.nao
    naux = auxmol.nao
    df_coef = scipy.linalg.solve(ints_2c2e, ints_3c2e.reshape(nao*nao, naux).T)
    df_coef = df_coef.reshape(naux, nao, nao)
    if dm1.ndim > 2:
        df_basis = []
        for j in range(dm1.shape[0]):
            df_basis.append(lib.einsum('Pij,ij->P', df_coef, dm1[j]))
        df_basis = np.stack(df_basis, axis=0)
        print(df_basis.shape)

    else:
        df_basis = lib.einsum('Pij,ij->P', df_coef, dm1)

    calc_dict['df_coeff'] = df_basis
    calc_dict['auxbasis'] = auxbasis
    # print('calc_dict', calc_dict)
    oe = mf.mo_energy
    hm = hf.get_fock(mf)
    calc_dict.update({"mo_energies":oe, "density_matrix": dm1,
                        "hamiltonian_matrix": hm})
    res.append(calc_dict)
    results.append(res)
    if i%10 == 0:
        np.save(save_path, results, allow_pickle=True)
np.save(save_path, results, allow_pickle=True)
npy_data = utils.calc_dict_to_npy(results, convert_forces=False, compress_atoms=False)
npy_data_compressed = utils.calc_dict_to_npy(results, convert_forces=False, compress_atoms=True)
# print('atom_number nc', npy_data['atom_numbers'][:3])
# print('atom_number c', npy_data_compressed['atom_numbers'][:3])
# print('pos nc', npy_data['positions'][:3])
# print('pos c', npy_data_compressed['positions'][:3])
# print('forces nc', npy_data['forces'][:3])
# print('forces c', npy_data_compressed['forces'][:3])
# print('forces sad nc', npy_data['forces_SAD'][:3])
# print('forces sad c', npy_data_compressed['forces_SAD'][:3])
np.save(npy_path, npy_data, allow_pickle=True)

18
results len 0
calc 0
converged SCF energy = -76.3593075295394
--------------- RKS gradients ---------------
         x                y                z
0 O    -0.0031590473    -0.0031590473     0.0000000000
1 H     0.0006956410     0.0024633614    -0.0000000000
2 H     0.0024633614     0.0006956410    -0.0000000000
----------------------------------------------
elapsed 1.1173691749572754
mo_coeff shape (41, 41)
ints3c2e shape (41, 41, 294)
ints2c2e shape (294, 294)
calc 1
converged SCF energy = -76.3593065872441
--------------- RKS gradients ---------------
         x                y                z
0 O    -0.0031607244    -0.0022351491    -0.0022351491
1 H     0.0006957014     0.0017420050     0.0017420050
2 H     0.0024637452     0.0004921051     0.0004921051
----------------------------------------------
elapsed 1.056481122970581
mo_coeff shape (41, 41)
ints3c2e shape (41, 41, 294)
ints2c2e shape (294, 294)
calc 2
converged SCF energy = -76.3593075295394
--------------- RKS gr

In [ ]:
set_types = ['train', 'valid', 'test']
for set_type in set_types:
    data1 = np.load('datasets/h2o_small_' + set_type + '_augccpvdz_df_augccpvqzjkfit.npy', allow_pickle=True)
    data2 = np.load('datasets/h2o_small_' + set_type + '_dft_augccpvdz_hm_dm_oe_calc.npy', allow_pickle=True)
    save_path = 'datasets/h2o_small_' + set_type + '_dft_augccpvdz_df_hm_dm_oe_calc.npy'
    for i in range(len(data1)):
        data2[i][1]['auxbasis'] = data1[i][1]['auxbasis']
        data2[i][1]['df_coeff'] = data1[i][1]['df_coeff']

    np.save(save_path, data2, allow_pickle=True)

In [17]:
results = np.load(save_path, allow_pickle=True)
for res in results:
    calc = res[1]
    mo_coeff = calc['mo_coeff']
    mo_en = calc['mo_energies']
    mol = gto.M(atom=res[0]['atom'], basis=basis)
    s1e = mol.intor('int1e_ovlp')
    ks = calc['hamiltonian_matrix'] 
    mf = dft.RKS(mol)
    moe_calc, mo_calc = mf.eig(ks, s1e)
    print('moe calc', moe_calc)
    print('mo en', mo_en)
    print('mo en error kcal', utils.hartree_to_kcal(np.mean(np.abs(moe_calc - mo_en))))
    print('mo en error hartree', np.mean(np.abs(moe_calc - mo_en)))

moe calc [-18.77941342  -0.94186483  -0.4782721   -0.35201599  -0.26678736
  -0.0359311    0.01966254   0.08871976   0.10821145   0.11597369
   0.14045355   0.16647297   0.21631347   0.26261142   0.26726919
   0.2984314    0.42980765   0.46432998   0.51202908   0.58817288
   0.72088599   0.86827459   0.88347438   0.91233964   1.00137169
   1.13292667   1.1551329    1.28143649   1.68199929   1.68887035
   1.81578292   1.99889631   2.03790594   2.26650384   2.36135552
   2.61382456   3.19210097   3.21456527   3.22549721   3.48752511
   3.83322557]
mo en [-18.77941873  -0.94186692  -0.47827374  -0.35201808  -0.2667895
  -0.03593132   0.01966234   0.08871947   0.10821105   0.11597344
   0.14045302   0.16647275   0.21631326   0.26261135   0.26726886
   0.29843109   0.42980748   0.46432973   0.51202841   0.5881725
   0.72088543   0.86827366   0.88347357   0.9123389    1.0013707
   1.13292509   1.15513185   1.28143522   1.68199911   1.68886989
   1.81578233   1.9988957    2.03790525   2.26650

moe calc [-1.87884821e+01 -9.35279070e-01 -4.46016977e-01 -3.71889217e-01
 -2.66379363e-01 -4.10718448e-02  1.83991453e-02  8.50149129e-02
  1.09346647e-01  1.16190937e-01  1.27068273e-01  1.64391582e-01
  2.27533179e-01  2.44588958e-01  2.66964642e-01  2.78998923e-01
  4.34523173e-01  4.56380051e-01  4.96063290e-01  5.67829414e-01
  7.29234154e-01  8.78250166e-01  8.83868350e-01  9.09687820e-01
  9.82587626e-01  1.10776189e+00  1.13372147e+00  1.27405286e+00
  1.65708455e+00  1.67815134e+00  1.80683039e+00  1.82924391e+00
  2.05230547e+00  2.19891537e+00  2.30471471e+00  2.67402328e+00
  3.19477162e+00  3.19878994e+00  3.21137592e+00  3.47201542e+00
  3.73703793e+00]
mo en [-1.87884891e+01 -9.35281663e-01 -4.46019078e-01 -3.71891947e-01
 -2.66382190e-01 -4.10721221e-02  1.83989019e-02  8.50145380e-02
  1.09346108e-01  1.16190528e-01  1.27067682e-01  1.64391256e-01
  2.27533061e-01  2.44588509e-01  2.66964619e-01  2.78998626e-01
  4.34523036e-01  4.56379682e-01  4.96062569e-01  5.67829

In [ ]:
results = []
for i in range(len(mfs)):
    #print(mfs[i].mo_coeff)
    mol_dict = mols[i].pack()
    calc_dict = {}
    calc_dict['mo_coeff'] = mfs[i].mo_coeff
    calc_dict['mo_occ'] = mfs[i].mo_occ
    calc_dict['energy'] = mfs[i].e_tot
    calc_dict['forces'] = forces[i]
    results.append((mol_dict, calc_dict))
    results.append(res)

np.save('datasets/h2o_dynamic_pyscf_631gss_dft_f.npy', results)

In [ ]:
data_scf = np.load('datasets/h2o_dynamic_pyscf_631gss_dft_f.npy', allow_pickle=True)
print(data_scf)
print(len(data_scf))

In [ ]:
data_scf = np.load('datasets/h2o_dynamic_pyscf_dft_f.npy', allow_pickle=True)
print(data_scf)
print(len(data_scf))
for d in data_scf:
    print('energy', d['energy'])
    print('forces', d['forces'])

In [ ]:
for d in data_scf:
    d['forces'] = d['forces'] * 0.529177


np.save('datasets/h2o_dynamic_pyscf_dft_f.npy', data_scf)

In [ ]:
new_data = []
data_scf = data_scf = np.load('datasets/h2o_dynamic_pyscf_dft_f.npy', allow_pickle=True)
for d in data_scf:
    new_d = []
    mo_coeff = d.pop('mo_coeff')
    mo_occ = d.pop('mo_occ')
    en = d.pop('energy')
    f = d.pop('forces')
    new_d.append(d)
    new_d.append({'mo_coeff': mo_coeff, 'mo_occ': mo_occ, 'energy': en, 'forces': f})
    new_data.append(new_d)

np.save('datasets/h2o_dynamic_pyscf_dft_f_en.npy', new_data)

In [ ]:
results = {'E': [], 'F': [], 'R': [], 'z': np.array([8, 1, 1])}
for i in range(len(mfs)):
    #print(mfs[i].mo_coeff)
    res = mols[i].pack()
    pos = []
    for a in res['atom']:
        pos.append(a[1])
    pos = np.array(pos)
    print('pos.shape', pos.shape)
    results['R'].append(pos)
    results['E'].append(mfs[i].e_tot)
    results['F'].append(forces[i])
    
results['R'] = np.array(results['R'])
results['E'] = np.array(results['E'])
results['F'] = np.array(results['F'])

np.savez('datasets/water_pyscf_dft_f', **results)

In [ ]:
results = {'energy': [], 'forces': [], 'positions': [],
           'atom_numbers': [8, 1, 1], 'atom_types': ['O', 'H', 'H'],
          'mo_coeff': [], 'mo_occ': []}
for d in data_scf:
    #print(mfs[i].mo_coeff)
    pos = []
    for a in d['atom']:
        pos.append(a[1])
    pos = np.array(pos)
    print('pos.shape', pos.shape)
    results['positions'].append(pos)
    results['energies'].append(d['energy'])
    results['forces'].append(d['forces'])
    results['mo_coeff'].append(d['mo_coeff'])
    results['mo_occ'].append(d['mo_occ'])

for key in results.keys():
    results[key] = np.array(results[key])
    
np.savez('datasets/h2o_dynamic_pyscf_dft_f', **results)

In [ ]:
print(np.load('datasets/h2o_dynamic_pyscf_dft.npy', allow_pickle=True)[0])
print(np.load('datasets/h2o_dynamic_pyscf_dft_f.npy', allow_pickle=True)[0])
print(np.load('datasets/h2o_dynamic_pyscf_dft_f_en.npy', allow_pickle=True)[0])